1. .enf: archivo que sirve para indicar al data manager que existe un registro. eclipse database handle
2. .system: archivo que tiene la información del sistema en el momento de grabación, es decir, el número de cámaras, los parámetros de cada una de esas cámaras, etc.
3. .xcp: archivo de calibración (camera configuration and settings).
4. .x1d: archivo que contiene los datos analógicos.
5. x2d: archivo que contiene los datos en bruto de las cámaras. raw capture data
6. vst and vsk - Vicon Skeleton file (t is a template, k is customized template)
7. .c3d: archivo que, una vez hecha la reconstrucción y etiquetado en nexus, contiene una combinación de los datos contenidos en los archivos .x1d, .x2d, .xcp y .system. contains only 3d point data
8. Raw camera data from Vicon V-series systems in .tvd format files.
9. Multimedia sound and moving picture data in .mpg or .avi format files.
10. Raw analog data (e.g. from force plates) in .vad format files.
11. Vicon subject kinematic data in .v format file
12.  Raw analog data (e.g. from force plates) in .vad format files.

Esto significa que, si no se van a procesar más los datos en nexus, podríamos guardar sólo el archivo .c3d en la carpeta preprocessed y posteriores  sin perder información. Lo que debemos tener en cuenta es que si en algún momento hay que volver a reconstruir y etiquetar esa captura, tendríamos que usar los datos de la carpeta raw data porque para poder hacerlo son necesarios los archivos .x2d y .xcp. Si quisiéramos seguir procesando, también puede hacerse con el .c3d siempre y cuando esa captura ya esté reconstruida y etiquetada. Para esto último tendríamos que guardar también el .enf y los archivos del sujeto (.mp y .vsk).

https://github.com/EmbodiedCognition/py-c3d/blob/faeaca1bb5cf8066a351ffc734cae0467bc7aa57/c3d/scripts/c3d_metadata.py#L44

In [54]:
import sys
sys.path.append('../preprocess-data-vicon/vicon_utils')
import utils

import importlib
importlib.reload(utils)

In [69]:
from pathlib import Path

# route_in = '/Volumes/AI4HA/PILOT_TRIALS/PREPROCESSED_RAW_DATA/SUBJECT_10/SESSION_01/GROSS_MOTOR_BIOMETRY/VICON/SUBJECT_10/SESSION_01'
# route_in = 'dinamic02c3d/'
route_in = 'ai4ha_in'
# route_out = '/Volumes/AI4HA/PILOT_TRIALS/EUROBENCH_FORMAT/SUBJECT_10/SESSION_01/GROSS_MOTOR_BIOMETRY/VICON/SUBJECT_10/SESSION_01'

file_in = 'subject_10_cond_11_run_01.c3d'
# file_in = 'dinamic02.c3d'
c3d_data = Path(route_in).joinpath(file_in)
c3d_data

### Read c3d and import trajectory data

In [70]:
import ezc3d

c = ezc3d.c3d(str(c3d_data))
print(c['parameters']['POINT']['USED']['value'][0])  # Print the number of points used
point_data = c['data']['points']
points_residuals = c['data']['meta_points']['residuals']
analog_data = c['data']['analogs']

In [71]:
c.keys()

In [72]:
c['header'].keys()
# c['header']['points'].keys()

In [73]:
c['header']['points'].keys()

In [74]:
# print all key-value pairs for c['header']
for key, value in c['header'].items():
    print('----- Key: ', key)
    # print key-value pairs for c['header']['key']
    for key_2, value_2 in value.items():
        print(key_2, ' : ', value_2)

In [75]:
# print all key-value pairs for c['header'] iteratively. Check that the type of the value is a dict
def print_dict(d):
    for key, value in d.items():
        print('----- Key: ', key)
        if type(value) == dict:
            print_dict(value)
        else:
            print(key, ' : ', value)
            
print_dict(c['header'])

In [76]:
# print all key-value pairs for c['header']
for key, value in c['parameters'].items():
    print('----- Key: ', key)
    # print key-value pairs for c['header']['key']
    for key_2, value_2 in value.items():
        print(key_2, ' : ', value_2)

In [77]:
c['parameters'].keys()

In [78]:
point_data.shape

In [79]:
points_residuals

In [80]:
analog_data

In [81]:
import c3d
from itertools import product

def print_metadata(reader):
    print('Header information:\n{}'.format(reader.header))
    for key, g in sorted(reader.group_items()):
        print('')
        for key, p in sorted(g.param_items()):
            print_param(g, p)


def print_param_value(name, value):
    print(name, '=', value)


def print_param_array(name, p, offset_in_elements):
    arr = []
    start = offset_in_elements
    end = offset_in_elements + p.dimensions[0]
    if p.bytes_per_element == 2:
        arr = p.int16_array
    elif p.bytes_per_element == 4:
        arr = p.float_array
    elif p.bytes_per_element == -1:
        return print_param_value(name, p.bytes[start:end])
    else:
        arr = p.int8_array
    print('{0} = {1}'.format(name, arr.flatten()[start:end]))


def print_param(g, p):
    name = "{0.name}.{1.name}".format(g, p)
    print('{0}: {1.total_bytes}B {1.dimensions}'.format(name, p))

    if len(p.dimensions) == 0:
        val = None
        width = len(p.bytes)
        if width == 2:
            val = p.int16_value
        elif width == 4:
            val = p.float_value
        else:
            val = p.int8_value
        print_param_value(name, val)

    if len(p.dimensions) == 1 and p.dimensions[0] > 0:
        return print_param_array(name, p, 0)

    if len(p.dimensions) >= 2:
        offset = 0
        for coordinate in product(*map(range, reversed(p.dimensions[1:]))):
            subscript = ''.join(["[{0}]".format(x) for x in coordinate])
            print_param_array(name + subscript, p, offset)
            offset += p.dimensions[0]

with open(c3d_data, 'rb') as handle:
    print_metadata(c3d.Reader(handle))
    for frame_no, points, analog in c3d.Reader(handle).read_frames(copy=False):
        fields = []

        print('frame {}: point {}, analog {}'.format(frame_no, points.shape, analog.shape))
        for x, y, z, err, cam in points:
            fields.append([str(x), str(y), str(z)])

In [68]:
fields

### First and second rows have to be concatenated

In [ ]:
df = utils.join_column_index_subindex(df)

### Clean the dataframe

In [ ]:
df = utils.clean_format_dataframe_trajectories(df, fs, units)

print(df.columns)

In [ ]:
df.head()

### Replace the column names in VICON format to the ones in the EUROBENCH format

In [ ]:
df = utils.rename_body_parts_trajectories(df)
df.head()

In [ ]:
df.columns

# Save data

In [ ]:
out_csv_name = path.Path(route_out).joinpath(path.Path(file_in).stem + '.csv')
# Save dataframe as csv
df.to_csv(out_csv_name)

# Signatures

## Input

- Multiindex csv with columns:
    ['LKNE', 'LTIB', 'LANK', 'LHEE', 'LTOE', 'RKNE', 'RTIB', 'RANK', 'RHEE',
       'RTOE', 'L_FOOTC1', 'L_FOOTC2', 'R_FOOTC1', 'R_FOOTC2']
- Subcolumn indexed with:
    ['Frame', 'Subframe', 'X', 'Y', 'Z']

## Output

- Single index table with columns:
    ['time', 'l_knee_x', 'l_knee_y', 'l_knee_z', 'l_tibia_x', 'l_tibia_y',
       'l_tibia_z', 'l_ankle_x', 'l_ankle_y', 'l_ankle_z', 'l_heel_x',
       'l_heel_y', 'l_heel_z', 'l_toe_x', 'l_toe_y', 'l_toe_z', 'r_knee_x',
       'r_knee_y', 'r_knee_z', 'r_tibia_x', 'r_tibia_y', 'r_tibia_z',
       'r_ankle_x', 'r_ankle_y', 'r_ankle_z', 'r_heel_x', 'r_heel_y',
       'r_heel_z', 'r_toe_x', 'r_toe_y', 'r_toe_z', 'l_footc1_x', 'l_footc1_y',
       'l_footc1_z', 'l_footc2_x', 'l_footc2_y', 'l_footc2_z', 'r_footc1_x',
       'r_footc1_y', 'r_footc1_z', 'r_footc2_x', 'r_footc2_y', 'r_footc2_z']



